# Chapter 01 — Apache Beam & Google Cloud Dataflow Fundamentals

Welcome to **Chapter 01** of the **Google Cloud Dataflow Workshop**!

In this interactive notebook, you will learn the foundational concepts of the **Apache Beam** unified programming model and see how pipelines run locally on your machine with **`DirectRunner`** and at enterprise scale on **Google Cloud Dataflow** with **`DataflowRunner`**.

## 1. What is Apache Beam?

**Apache Beam** (combining **B**atch + Str**eam**) is an open-source, unified programming model for defining data processing pipelines.

* **Pipeline as Code**: You define the directed acyclic graph (DAG) of data transformations in Python, Java, Go, or YAML.
* **Write Once, Run Anywhere**: The pipeline logic is completely decoupled from the underlying execution engine. The exact same Beam code can run on:
  * **`DirectRunner`**: Runs locally on your workstation for instant feedback, rapid debugging, and unit testing.
  * **`DataflowRunner`**: Google Cloud's fully managed, serverless, horizontally auto-scaling cloud runtime.
  * **`SparkRunner` / `FlinkRunner`**: Apache Spark or Apache Flink clusters on-premises or multi-cloud.

### The Core Beam Building Blocks
1. **`Pipeline`**: The root container managing your execution graph and runtime options.
2. **`PCollection`**: A distributed, immutable dataset representing data inside the pipeline (either bounded batch or unbounded stream).
3. **`PTransform`**: A processing operation applied to one or more `PCollections` using the pipe (`|`) operator.
4. **`DoFn` / `beam.ParDo`**: User-defined element-wise transformation functions (parallel Do).
5. **`Runner`**: The execution engine that compiles and runs the Beam pipeline graph.

## 2. Environment Setup & Activating the Kernel with `uv`

This project uses **[`uv`](https://docs.astral.sh/uv/)** to manage dependencies in an isolated `.venv` virtual environment.

### How to Activate the Environment

#### Option A: In the Terminal
If you are running scripts from the command line, activate the environment with:
```bash
# macOS / Linux:
source .venv/bin/activate

# Or execute any command directly without manual activation:
uv run python pipelines/sales_pipeline.py --help
```

#### Option B: In This Jupyter Notebook (Kernel Selection)
If you run a cell and see `ModuleNotFoundError: No module named 'apache_beam'`, the notebook is attached to your global system Python instead of the `.venv`.

1. **Register the kernel** (run this once from your terminal in the repository root):
   ```bash
   uv run python -m ipykernel install --user --name dataflow-workshops --display-name "Python (dataflow-workshops)"
   ```
2. **Select the kernel in your IDE**:
   * Look at the top-right corner of this notebook editor window.
   * Click the **Kernel Picker** button.
   * Select **`Python (dataflow-workshops)`** or browse to **`.venv/bin/python`**.

--- 

### Verification Cell
Run the cell below. If it executes without error and displays an interpreter path containing `.venv`, you are correctly attached to the project environment!

In [ ]:
import os
import sys

# 1. Verify Python executable location
print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is attached to the project's isolated .venv environment!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Please select 'Python (dataflow-workshops)' from the top-right kernel picker.")

# 2. Import and verify Apache Beam
try:
    import apache_beam as beam
    print(f"✅ Apache Beam Version: {beam.__version__}")
except ImportError as e:
    print(f"❌ Failed to import apache_beam: {e}")
    print("Run 'uv sync' in your terminal and select the registered kernel.")

## 3. Your First In-Memory Beam Pipeline (`beam.Create`)

To understand how Beam constructs and executes a Directed Acyclic Graph (DAG), we will create an in-memory `PCollection` using `beam.Create()`.

### The Anatomy of Beam Pipeline Syntax

In Apache Beam, pipelines are constructed using the pipe operator (`|`) and the right-shift operator (`>>`):

```python
[Output PCollection] = [Input PCollection] | "Unique Transform Name" >> [PTransform]
```

| Component | Type | Visual Shape | Description |
| :--- | :--- | :--- | :--- |
| **`PCollection`** | **Data** | 🛢️ Cylinder / Oval `[( ... )]` | An immutable, distributed collection of elements (bounded or unbounded). |
| **`PTransform`** | **Operation** | ⚙️ Rectangle `[ ... ]` | A processing operation that reads from an input `PCollection` and writes to an output `PCollection`. |
| **Pipe (`\|`)** | **Application** | $\rightarrow$ Arrow | Applies a transformation to the preceding collection. |
| **Shift (`>>`)** | **Naming** | Label binding | Attaches a unique human-readable label to the transform node. |

--- 

### What Does the "Unique Transform Name" Stand For?

In `p | "FilterEven" >> beam.Filter(...)`, the string `"FilterEven"` is not just a comment—it is a critical runtime identifier:

1. **DAG Graph Node Label**: Apache Beam compiles your code into an abstract execution graph. This string becomes the name of that node in the graph.
2. **Google Cloud Dataflow Console UI**: When deployed to Google Cloud, Dataflow renders an interactive visual pipeline graph. Every stage box in the console is labeled with this exact name, allowing you to visually inspect throughput, execution stages, and execution duration.
3. **Cloud Monitoring & Cloud Logging Metrics**: All Dataflow metrics (wall time, CPU utilization, element counters, throughput, and custom `beam.metrics`) are scoped by this name (e.g. `dataflow.googleapis.com/job/element_count` tagged with `step="FilterEven"`).
4. **Uniqueness Constraint**: Every transform name **must be unique** within its scope. If you declare two steps with the same name, Beam throws a compilation error:
   ```text
   ValueError: A transform with name "FilterEven" already exists in this pipeline
   ```
5. **Streaming Checkpoints & Pipeline Updates**: In production streaming pipelines, Dataflow tracks in-flight window state and timers keyed by the transform's full hierarchical name. When performing zero-downtime pipeline updates (`--update`), matching names allows Dataflow to preserve state across new code deployments.

--- 

### Pipeline Execution Graph (DAG)

Notice how the **`PCollection`** (data in motion, cylinder shape) and **`PTransform`** (processing steps, rectangle box) alternate in the execution graph:

![In-Memory Numbers Pipeline DAG](../artifacts/01_numbers_pipeline_dag.png)

<details>
<summary><b>Click to expand raw Mermaid flowchart definition</b></summary>

```mermaid
flowchart TD
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef legend fill:#ffffff,stroke:#dadce0,stroke-width:1px;

    subgraph Legend["Shape & Concept Legend"]
        direction LR
        LEG_PC[("[( Cylinder / Oval )]<br/>PCollection (Data in Motion)")]:::pcoll
        LEG_PT["[[ Rectangle ]]<br/>PTransform (Processing Logic)"]:::ptrans
    end

    PIPE[("p (Root Pipeline Object)")]:::pcoll

    T_CREATE["PTransform: beam.Create<br/>Name: 'CreateNumbers'"]:::ptrans
    PC_ALL[("PCollection&lt;int&gt;<br/>[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]")]:::pcoll

    T_FILTER["PTransform: beam.Filter(is_even)<br/>Name: 'FilterEven'"]:::ptrans
    PC_EVENS[("PCollection&lt;int&gt; (Filtered)<br/>[2, 4, 6, 8, 10]")]:::pcoll

    T_MAP["PTransform: beam.Map(n * 10)<br/>Name: 'MultiplyByTen'"]:::ptrans
    PC_MULT[("PCollection&lt;int&gt; (Transformed)<br/>[20, 40, 60, 80, 100]")]:::pcoll

    T_PRINT["PTransform: beam.Map(print)<br/>Name: 'PrintResults'"]:::ptrans
    PC_OUT[("Console Stdout Output")]:::pcoll

    PIPE --> T_CREATE
    T_CREATE --> PC_ALL
    PC_ALL --> T_FILTER
    T_FILTER --> PC_EVENS
    PC_EVENS --> T_MAP
    T_MAP --> PC_MULT
    PC_MULT --> T_PRINT
    T_PRINT --> PC_OUT
```
</details>

Now let's execute this pipeline with Apache Beam:

In [10]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

with beam.Pipeline() as p:
    (
        p
        | "CreateNumbers" >> beam.Create(numbers)
        | "FilterEven" >> beam.Filter(lambda n: n % 2 == 0)
        | "MultiplyByTen" >> beam.Map(lambda n: n * 10)
        | "PrintResults" >> beam.Map(lambda n: print(f"Even * 10: {n}"))
    )

Even * 10: 20
Even * 10: 40
Even * 10: 60
Even * 10: 80
Even * 10: 100


> **Key Takeaway**: When entering the `with beam.Pipeline() as p:` context, Beam constructs the abstract DAG graph. The pipeline is evaluated and executed only when the context exits!

## 4. Reading External Data: The Retail Sales Dataset

In our workshop storyline, we process a retail transactions dataset (`data/sales_data.csv`).

Let's inspect the first few lines of the CSV file using Python.

In [12]:
from pathlib import Path

csv_path = Path("data/sales_data.csv")
print(csv_path.read_text())

transaction_id,product_id,amount,transaction_date
TID001,PROD_A,150.75,2025-06-12
TID002,PROD_B,200.00,2025-06-12
TID003,PROD_A,75.20,2025-06-11
TID004,PROD_C,300.50,2025-06-12
TID005,PROD_B,120.99,2025-06-13
TID006,PROD_A,99.99,2025-06-12
TID007,PROD_D,50.00,2025-06-11
TID008,PROD_C,450.00,2025-06-13
TID009,PROD_A,25.00,2025-06-12
TID010,PROD_B,310.25,2025-06-12



Each row contains:
* `transaction_id`: Unique identifier (e.g., `TID001`)
* `product_id`: Product SKU (e.g., `PROD_A`)
* `amount`: Revenue in USD (e.g., `150.75`)
* `transaction_date`: Date string in `YYYY-MM-DD` format

Now let's use `beam.io.ReadFromText` to read these rows into a `PCollection`.

In [11]:
with beam.Pipeline() as p:
    (
        p
        | "ReadLines" >> beam.io.ReadFromText("data/sales_data.csv")
        | "PrintRawLines" >> beam.Map(lambda line: print(f"Raw line: {line}"))
    )

Raw line: transaction_id,product_id,amount,transaction_date
Raw line: TID001,PROD_A,150.75,2025-06-12
Raw line: TID002,PROD_B,200.00,2025-06-12
Raw line: TID003,PROD_A,75.20,2025-06-11
Raw line: TID004,PROD_C,300.50,2025-06-12
Raw line: TID005,PROD_B,120.99,2025-06-13
Raw line: TID006,PROD_A,99.99,2025-06-12
Raw line: TID007,PROD_D,50.00,2025-06-11
Raw line: TID008,PROD_C,450.00,2025-06-13
Raw line: TID009,PROD_A,25.00,2025-06-12
Raw line: TID010,PROD_B,310.25,2025-06-12


## 5. Custom Element-Wise Processing: `DoFn` and `beam.ParDo`

While simple inline transformations can use `beam.Map` or `beam.Filter` with lambdas, production data engineering requires structured, testable, and reusable transformation classes.

In Apache Beam, this is achieved with **`beam.ParDo`** and a custom **`DoFn`** (Do Function):
* Subclass `beam.DoFn`
* Implement the `process(self, element)` generator method
* Use `yield` to emit 0, 1, or multiple output elements (enabling filtering, mapping, or 1-to-many explosions)

Let's create two `DoFn` classes:
1. `ParseSalesRecordFn`: Ignores the header row and parses CSV strings into Python dictionaries.
2. `FilterByDateFn`: Filters transactions for a specific date (e.g. `2025-06-12`).

In [ ]:
class ParseSalesRecordFn(beam.DoFn):
    """Parses a CSV row into a structured dictionary, skipping headers."""
    def process(self, line: str):
        line = line.strip()
        if not line or line.startswith("transaction_id"):
            return  # Skip empty lines and CSV header
        
        parts = [p.strip() for p in line.split(",")]
        if len(parts) == 4:
            try:
                yield {
                    "transaction_id": parts[0],
                    "product_id": parts[1],
                    "amount": float(parts[2]),
                    "transaction_date": parts[3],
                }
            except ValueError:
                pass

class FilterByDateFn(beam.DoFn):
    """Filters transactions matching a target date."""
    def __init__(self, target_date: str):
        self.target_date = target_date
        
    def process(self, record: dict):
        if record.get("transaction_date") == self.target_date:
            yield record

Let's test our `DoFn` classes inside a pipeline:

In [ ]:
with beam.Pipeline() as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("data/sales_data.csv")
        | "ParseRecords" >> beam.ParDo(ParseSalesRecordFn())
        | "FilterDate" >> beam.ParDo(FilterByDateFn(target_date="2025-06-12"))
        | "PrintFiltered" >> beam.Map(lambda r: print(f"Filtered: {r}"))
    )

## 6. Aggregations: `GroupByKey` vs. `CombinePerKey`

When aggregating distributed data (e.g., calculating total revenue per `product_id`):

| Feature | `beam.GroupByKey` | `beam.CombinePerKey` |
| :--- | :--- | :--- |
| **Shuffle Cost** | High (transfers every raw element across the network) | Low (partial combiner reduces data locally before network shuffle) |
| **Memory Pressure** | High (can cause Out-Of-Memory errors on skewed keys) | Low (constant memory per key on each worker) |
| **Requirement** | Collects an iterable of all raw values | Aggregation function must be associative and commutative (like `sum`, `min`, `max`) |

> **FinOps & Performance Best Practice**: Always prefer **`beam.CombinePerKey`** over `beam.GroupByKey` whenever performing commutative mathematical reductions (sum, count, min, max)!

Let's map our parsed records to `(product_id, amount)` tuples and compute total revenue with `beam.CombinePerKey(sum)`.

In [ ]:
with beam.Pipeline() as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("data/sales_data.csv")
        | "ParseRecords" >> beam.ParDo(ParseSalesRecordFn())
        | "FilterDate" >> beam.ParDo(FilterByDateFn(target_date="2025-06-12"))
        | "MapKeyVal" >> beam.Map(lambda r: (r["product_id"], r["amount"]))
        | "SumRevenue" >> beam.CombinePerKey(sum)
        | "PrintSummary" >> beam.Map(lambda kv: print(f"Product: {kv[0]} -> Total Revenue: ${kv[1]:.2f}"))
    )

## 7. Writing Results: `beam.io.WriteToText`

In production, results are written to partitioned files on **Cloud Storage (`gs://`)** or tables in **BigQuery**.

Here is the complete DAG of our end-to-end retail pipeline, showing how **`PCollection`** (data cylinders) and **`PTransform`** (operation boxes) connect:

![Retail Sales Pipeline DAG](../artifacts/01_sales_pipeline_dag.png)

<details>
<summary><b>Click to expand raw Mermaid flowchart definition</b></summary>

```mermaid
flowchart TD
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;

    CSV[("data/sales_data.csv<br/>(Local Disk / GCS)")]:::pcoll

    T_READ["PTransform: beam.io.ReadFromText<br/>Name: 'ReadCSV'"]:::ptrans
    PC_RAW[("PCollection&lt;str&gt;<br/>(Raw CSV lines)")]:::pcoll

    T_PARSE["PTransform: beam.ParDo(ParseSalesRecordFn)<br/>Name: 'ParseRecords'"]:::ptrans
    PC_PARSED[("PCollection&lt;dict&gt;<br/>(Parsed sales records)")]:::pcoll

    T_FILTER["PTransform: beam.ParDo(FilterByDateFn)<br/>Name: 'FilterDate'"]:::ptrans
    PC_FILTERED[("PCollection&lt;dict&gt;<br/>(Records for 2025-06-12)")]:::pcoll

    T_MAP["PTransform: beam.Map(lambda r: (product_id, amount))<br/>Name: 'MapKeyVal'"]:::ptrans
    PC_KV[("PCollection&lt;tuple&gt;<br/>(product_id, amount) pairs")]:::pcoll

    T_COMBINE["PTransform: beam.CombinePerKey(sum)<br/>Name: 'SumRevenue'"]:::ptrans
    PC_AGG[("PCollection&lt;tuple&gt;<br/>(product_id, total_revenue)")]:::pcoll

    T_FORMAT["PTransform: beam.Map(format_output)<br/>Name: 'FormatOutput'"]:::ptrans
    PC_STR[("PCollection&lt;str&gt;<br/>(Formatted output strings)")]:::pcoll

    T_WRITE["PTransform: beam.io.WriteToText<br/>Name: 'WriteResults'"]:::ptrans
    OUT_FILES[("output/notebook_sales_summary-*.txt<br/>(Partitioned Text Files)")]:::pcoll

    CSV --> T_READ
    T_READ --> PC_RAW
    PC_RAW --> T_PARSE
    T_PARSE --> PC_PARSED
    PC_PARSED --> T_FILTER
    T_FILTER --> PC_FILTERED
    PC_FILTERED --> T_MAP
    T_MAP --> PC_KV
    PC_KV --> T_COMBINE
    T_COMBINE --> PC_AGG
    PC_AGG --> T_FORMAT
    T_FORMAT --> PC_STR
    PC_STR --> T_WRITE
    T_WRITE --> OUT_FILES
```
</details>

Now let's execute the full pipeline:

In [ ]:
output_prefix = "output/notebook_sales_summary"

with beam.Pipeline() as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("data/sales_data.csv")
        | "ParseRecords" >> beam.ParDo(ParseSalesRecordFn())
        | "FilterDate" >> beam.ParDo(FilterByDateFn(target_date="2025-06-12"))
        | "MapKeyVal" >> beam.Map(lambda r: (r["product_id"], r["amount"]))
        | "SumRevenue" >> beam.CombinePerKey(sum)
        | "FormatOutput" >> beam.Map(lambda kv: f"product_id={kv[0]}, total_revenue={kv[1]:.2f}")
        | "WriteResults" >> beam.io.WriteToText(output_prefix, file_name_suffix=".txt")
    )

print("Pipeline completed successfully!")

In [ ]:
# Let's inspect the generated file(s)
for file in Path("output").glob("notebook_sales_summary*.txt"):
    print(f"--- Contents of {file.name} ---")
    print(file.read_text())

## 8. From Local (`DirectRunner`) to Cloud Dataflow (`DataflowRunner`)

So far, we have executed everything locally using `DirectRunner`.

To deploy this exact pipeline to **Google Cloud Dataflow**, you only need to change the **`PipelineOptions`**:

```python
from apache_beam.options.pipeline_options import PipelineOptions

options = PipelineOptions([
    "--runner=DataflowRunner",                  # Switch from DirectRunner to Dataflow
    "--project=YOUR_GCP_PROJECT_ID",            # Google Cloud Project ID
    "--region=europe-west1",                    # Google Cloud Region for worker VMs
    "--job_name=chapter01-sales-pipeline",      # Name displayed in GCP Console
    "--temp_location=gs://YOUR_BUCKET/temp",    # GCS bucket for temporary files
    "--staging_location=gs://YOUR_BUCKET/stg", # GCS bucket for staged binary packages
    "--input=gs://YOUR_BUCKET/sales_data.csv",  # Read from Cloud Storage
    "--output=gs://YOUR_BUCKET/output/summary", # Write to Cloud Storage
    "--save_main_session",                      # Serializes global imports to remote workers
])

with beam.Pipeline(options=options) as p:
    # Exact same transformation code runs on Google Cloud!
    ...
```

### How Dataflow Executes Your Pipeline
1. **Graph Construction & Submission**: The Beam SDK builds the DAG and serializes it as a JSON execution specification to the Dataflow API (`dataflow.googleapis.com`).
2. **Graph Optimization & Fusion**: Dataflow analyzes the DAG and fuses compatible adjacent transforms (e.g., `Read` + `Parse` + `Filter` + `Map`) into a single execution step to eliminate intermediate serialization overhead.
3. **Worker Provisioning**: Dataflow dynamically spins up worker Compute Engine VMs in your VPC subnetwork.
4. **Dynamic Work Rebalancing**: As workers consume data, Dataflow monitors progress and automatically rebalances work items to eliminate stragglers.
5. **Automatic Teardown**: As soon as the last record is written, Dataflow deletes the worker VMs and finalizes job state.

## 9. Running via Modular Scripts

We have packaged the entire pipeline into a production-ready driver script [`pipelines/sales_pipeline.py`](./pipelines/sales_pipeline.py).

You can execute it using the accompanying shell scripts:

```bash
# 1. Run locally with DirectRunner:
bash 01_beam_fundamentals/scripts/01_run_local.sh

# 2. Deploy to Google Cloud Dataflow:
bash 01_beam_fundamentals/scripts/02_run_dataflow.sh
```

---

## Summary & What's Next

In this chapter, you learned:
* The **Apache Beam unified execution model** (`Pipeline`, `PCollection`, `PTransform`).
* How to parse raw CSV data and filter rows using `DoFn` and `beam.ParDo`.
* Why **`beam.CombinePerKey`** is the gold standard for high-performance aggregations.
* How to switch seamlessly between local development (`DirectRunner`) and cloud production (`DataflowRunner`).

👉 **Next: [Chapter 02 — Advanced Transformations & BigQuery I/O](../02_transforms_and_bigquery/README.md)**:
We will explore **Side Inputs** for dynamic lookups, **Tagged Multi-Outputs** (dead-letter branches for poison records), and high-throughput streaming/batch writes using the **BigQuery Storage Write API**!